# RUN 04 — `load_dataset` + FinBERT + feature ablation

Версия RUN 04 в формате Jupyter Notebook, где данные загружаются как раньше через:

```python
from datasets import load_dataset
```

Ноутбук делает полный pipeline:

1. загружает `TheFinAI/flare-sm-bigdata`;
2. достаёт твиты по тикеру;
3. строит FinBERT embeddings;
4. загружает OHLCV через `yfinance`;
5. собирает feature dataset;
6. запускает RUN 04 feature ablation;
7. сохраняет результаты в `artifacts/run_04_load_dataset/`.

## 0. Установка пакетов, если нужно

Если запускаешь локально и всё уже стоит, ячейку можно пропустить.

In [ ]:
# %pip install -q datasets transformers yfinance scikit-learn torch pandas numpy

## 1. Импорты и настройки

In [ ]:
from pathlib import Path
import sys
import re
import json
import random
import importlib.util

import numpy as np
import pandas as pd
from IPython.display import display

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import yfinance as yf

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RUN04_PATH = ROOT / 'experiments' / 'run_04_feature_ablation.py'
assert RUN04_PATH.exists(), f'Не найден файл: {RUN04_PATH}'
spec = importlib.util.spec_from_file_location('run04', RUN04_PATH)
run04 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run04)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TICKER = 'aapl'
TICKER_YF = TICKER.upper()
HF_DATASET = 'TheFinAI/flare-sm-bigdata'
SPLITS = ['train', 'validation', 'test']

TEXT_LAG_DAYS = 1
MAX_TEXT_WINDOW_DAYS = 10
BATCH_SIZE = 16
EMB_PREFIX = 'emb_'

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
PCA_COMPONENTS = (16, 32, 64)
C_GRID = (0.01, 0.1, 1.0, 10.0)

OUTPUT_DIR = ROOT / 'artifacts' / 'run_04_load_dataset'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_OUT = OUTPUT_DIR / 'run04_feature_dataset.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('ROOT:', ROOT)
print('device:', device)
print('ticker:', TICKER_YF)
print('output:', OUTPUT_DIR)

## 2. Загрузка твитов через `load_dataset`

Логика такая же, как в предыдущем ноутбуке: берём `TheFinAI/flare-sm-bigdata`, фильтруем по `$aapl`, достаём строки с датами.

In [ ]:
def extract_tweets_with_dates(text_block: str, ticker: str):
    if not isinstance(text_block, str):
        return []
    rows = []
    pattern = re.compile(r'^(\d{4}-\d{2}-\d{2}):\s*(.*)$')
    for line in text_block.split('\n'):
        line = line.strip()
        if not line:
            continue
        match = pattern.match(line)
        if not match:
            continue
        date_str, tweet = match.group(1), match.group(2).strip()
        if tweet and f'${ticker}' in tweet.lower():
            rows.append({'text_date': date_str, 'tweet': tweet})
    return rows

all_tweets = []
print(f'Загрузка датасета {HF_DATASET}...')

for split in SPLITS:
    try:
        ds = load_dataset(HF_DATASET, split=split)
        print(f'  {split}: {len(ds)} записей')
        for example in ds:
            query = example.get('query', '')
            text = example.get('text', '')
            if f'${TICKER}' not in str(query).lower() and f'${TICKER}' not in str(text).lower():
                continue
            all_tweets.extend(extract_tweets_with_dates(text, TICKER))
    except Exception as e:
        print(f'  Не удалось загрузить split={split}: {e}')

tweets = pd.DataFrame(all_tweets).drop_duplicates()
if tweets.empty:
    raise ValueError(f'Не найдено твитов для ${TICKER}')

tweets['text_date'] = pd.to_datetime(tweets['text_date'])
tweets = tweets.sort_values('text_date').reset_index(drop=True)
print('tweets:', tweets.shape)
print('date range:', tweets['text_date'].min(), '—', tweets['text_date'].max())
display(tweets.head())
tweets.to_csv(OUTPUT_DIR / 'prepared_tweets.csv', index=False)

## 3. FinBERT embeddings

Используем `ProsusAI/finbert`. Для каждого твита берём CLS embedding размерности 768.

In [ ]:
model_name = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert = AutoModel.from_pretrained(model_name).to(device)
finbert.eval()

def embed_texts(texts, batch_size=BATCH_SIZE):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            out = finbert(**encoded)
            cls = out.last_hidden_state[:, 0, :].detach().cpu().numpy()
        vectors.append(cls)
    return np.vstack(vectors)

emb_cache = OUTPUT_DIR / 'tweet_embeddings.npy'
if emb_cache.exists():
    embeddings = np.load(emb_cache)
    print('loaded cached embeddings:', embeddings.shape)
else:
    embeddings = embed_texts(tweets['tweet'].tolist())
    np.save(emb_cache, embeddings)
    print('saved embeddings:', embeddings.shape)

emb_cols = [f'{EMB_PREFIX}{i}' for i in range(embeddings.shape[1])]
emb_df = pd.DataFrame(embeddings, columns=emb_cols)
tweets_emb = pd.concat([tweets.reset_index(drop=True), emb_df], axis=1)
display(tweets_emb.head())

## 4. Загрузка OHLCV через `yfinance` и сбор feature dataset

Для каждой торговой даты берём тексты не позднее `t-1` и не старше `MAX_TEXT_WINDOW_DAYS`.

In [ ]:
price_start = (tweets['text_date'].min() - pd.Timedelta(days=10)).strftime('%Y-%m-%d')
price_end = (tweets['text_date'].max() + pd.Timedelta(days=120)).strftime('%Y-%m-%d')

prices = yf.download(TICKER_YF, start=price_start, end=price_end, auto_adjust=False, progress=False)
if prices.empty:
    raise ValueError('yfinance не вернул данные')

if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = [c[0] for c in prices.columns]
prices = prices.reset_index()
prices.columns = [str(c).lower().replace(' ', '_') for c in prices.columns]
prices = prices.rename(columns={'date': 'decision_date', 'adj_close': 'adj-close'})
prices['decision_date'] = pd.to_datetime(prices['decision_date'])
prices = prices.sort_values('decision_date').reset_index(drop=True)

# target: next-day direction
prices['y'] = (prices['close'].shift(-1) > prices['close']).astype(float)

# old-style increment features
for k in [5, 10, 15, 20, 25, 30]:
    prices[f'inc-{k}'] = prices['close'] / prices['close'].shift(k) - 1

# daily mean embeddings first
daily_emb = tweets_emb.groupby('text_date')[emb_cols].mean().sort_index()
daily_count = tweets_emb.groupby('text_date').size().rename('daily_text_count')

rows = []
for _, row in prices.iterrows():
    decision_date = row['decision_date']
    window_end = decision_date - pd.Timedelta(days=TEXT_LAG_DAYS)
    window_start = decision_date - pd.Timedelta(days=MAX_TEXT_WINDOW_DAYS)
    window_dates = daily_emb.loc[(daily_emb.index >= window_start) & (daily_emb.index <= window_end)]
    if window_dates.empty:
        emb_mean = np.zeros(len(emb_cols), dtype=float)
        text_count = 0
    else:
        emb_mean = window_dates.mean(axis=0).values
        text_count = int(daily_count.loc[window_dates.index].sum())
    out = row.to_dict()
    out['window_text_count'] = text_count
    out['window_has_text'] = int(text_count > 0)
    out.update({col: val for col, val in zip(emb_cols, emb_mean)})
    rows.append(out)

dataset = pd.DataFrame(rows)
dataset = dataset.dropna(subset=['y']).reset_index(drop=True)
dataset['y'] = dataset['y'].astype(int)
dataset.to_csv(DATASET_OUT, index=False)

print('dataset:', dataset.shape)
print('saved:', DATASET_OUT)
display(dataset.head())
display(dataset[['decision_date', 'close', 'volume', 'y', 'window_text_count', 'window_has_text']].head(20))

## 5. RUN 04 feature groups + PCA

Теперь запускаем ablation на собранном датасете.

In [ ]:
df = dataset.copy()
date_col = 'decision_date'
target_col = 'y'
df.attrs['date_col'] = date_col

df, market_groups = run04.make_market_features(df, date_col)
df = df.sort_values(date_col).reset_index(drop=True)
df.attrs['date_col'] = date_col

embedding_cols = [c for c in df.columns if str(c).startswith(EMB_PREFIX)]
text_meta_cols = ['window_text_count', 'window_has_text']
df['embedding_norm'] = np.linalg.norm(df[embedding_cols].fillna(0).values, axis=1)
text_meta_cols.append('embedding_norm')

train_idx, val_idx, test_idx = run04.chronological_split(df, date_col, TRAIN_RATIO, VAL_RATIO)
all_idx = df.index.to_numpy()

feature_groups = {
    'prices_raw': market_groups.get('prices_raw', []),
    'returns_rolling': market_groups.get('returns_rolling', []),
    'volume_rolling': market_groups.get('volume_rolling', []),
    'market_full': market_groups.get('market_full', []),
    'finbert_full': embedding_cols,
    'text_meta': text_meta_cols,
    'market_full_text_meta': list(dict.fromkeys(market_groups.get('market_full', []) + text_meta_cols)),
}

pca_info = {}
for k in PCA_COMPONENTS:
    pca_df, pca_cols, info = run04.make_pca_features(df, embedding_cols, train_idx, all_idx, k, SEED)
    for col in pca_cols:
        df[col] = pca_df[col]
    pca_info[f'finbert_pca_{k}'] = info
    feature_groups[f'finbert_pca_{k}'] = pca_cols
    feature_groups[f'market_full_finbert_pca_{k}'] = list(dict.fromkeys(feature_groups['market_full'] + pca_cols))
    feature_groups[f'market_full_finbert_pca_{k}_text_meta'] = list(dict.fromkeys(feature_groups['market_full'] + pca_cols + text_meta_cols))

split_summary = pd.DataFrame([
    {'split': 'train', 'n': len(train_idx), 'date_min': df.iloc[train_idx][date_col].min(), 'date_max': df.iloc[train_idx][date_col].max(), 'positive_share': df.iloc[train_idx][target_col].mean()},
    {'split': 'validation', 'n': len(val_idx), 'date_min': df.iloc[val_idx][date_col].min(), 'date_max': df.iloc[val_idx][date_col].max(), 'positive_share': df.iloc[val_idx][target_col].mean()},
    {'split': 'test', 'n': len(test_idx), 'date_min': df.iloc[test_idx][date_col].min(), 'date_max': df.iloc[test_idx][date_col].max(), 'positive_share': df.iloc[test_idx][target_col].mean()},
])
display(split_summary)
display(pd.DataFrame([{'feature_group': k, 'n_features': len(v)} for k, v in feature_groups.items()]).sort_values('n_features', ascending=False))
print(json.dumps(pca_info, indent=2, ensure_ascii=False))

## 6. Обучение и оценка

`C` и threshold выбираются только по validation. Test используется только для финальной оценки.

In [ ]:
metrics_rows, cm_rows, prediction_frames, selection_rows = [], [], [], []

# Baselines
for group_name, evaluator in [
    ('majority', lambda: run04.evaluate_majority(df, target_col, train_idx, test_idx)),
    ('previous_direction', lambda: run04.evaluate_previous_direction(df, date_col, target_col, test_idx)),
]:
    m, cm, preds, details = evaluator()
    metrics_rows.append({'model': 'baseline', 'feature_group': group_name, **m, **details})
    cm_rows.append({'model': 'baseline', 'feature_group': group_name, **cm})
    preds['model'] = 'baseline'
    preds['feature_group'] = group_name
    prediction_frames.append(preds)

# Logistic Regression
for group_name, cols in feature_groups.items():
    cols = [c for c in cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    if not cols:
        print('skip empty group:', group_name)
        continue
    print('run:', group_name, 'n_features=', len(cols))
    m, cm, preds, details = run04.fit_logreg_group(
        df=df,
        feature_cols=cols,
        target_col=target_col,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        c_grid=C_GRID,
        random_state=SEED,
    )
    metrics_rows.append({'model': 'LogisticRegression', 'feature_group': group_name, **m, **details})
    cm_rows.append({'model': 'LogisticRegression', 'feature_group': group_name, **cm})
    selection_rows.append({'feature_group': group_name, **details})
    preds['model'] = 'LogisticRegression'
    preds['feature_group'] = group_name
    prediction_frames.append(preds)

metrics = pd.DataFrame(metrics_rows).sort_values(['balanced_accuracy', 'accuracy'], ascending=False)
confusion = pd.DataFrame(cm_rows)
selections = pd.DataFrame(selection_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

display(metrics)
display(confusion)
display(selections)

## 7. Delta-анализ относительно `market_full`

In [ ]:
metrics_delta = metrics.copy()
market_rows = metrics_delta[metrics_delta['feature_group'] == 'market_full']
if len(market_rows) > 0:
    market = market_rows.iloc[0]
    for col in ['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']:
        metrics_delta[f'delta_vs_market_full_{col}'] = metrics_delta[col] - market[col]

interesting = metrics_delta[metrics_delta['feature_group'].str.contains('majority|previous|market_full|finbert', regex=True)].copy()
display(interesting.sort_values(['balanced_accuracy', 'accuracy'], ascending=False))

## 8. Сохранение артефактов

После запуска пришли мне `metrics.csv`, `dataset_summary.json`, `validation_selection.csv`.

In [ ]:
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
confusion.to_csv(OUTPUT_DIR / 'confusion_matrix.csv', index=False)
selections.to_csv(OUTPUT_DIR / 'validation_selection.csv', index=False)
predictions.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)

with open(OUTPUT_DIR / 'feature_groups.json', 'w', encoding='utf-8') as f:
    json.dump({k: list(map(str, v)) for k, v in feature_groups.items()}, f, ensure_ascii=False, indent=2)

summary = {
    'hf_dataset': HF_DATASET,
    'ticker': TICKER_YF,
    'text_lag_days': TEXT_LAG_DAYS,
    'max_text_window_days': MAX_TEXT_WINDOW_DAYS,
    'n_tweets': int(len(tweets)),
    'n_observations': int(len(df)),
    'n_unique_dates': int(df[date_col].nunique()),
    'date_min': str(df[date_col].min().date()),
    'date_max': str(df[date_col].max().date()),
    'train_size': int(len(train_idx)),
    'validation_size': int(len(val_idx)),
    'test_size': int(len(test_idx)),
    'train_date_min': str(df.iloc[train_idx][date_col].min().date()),
    'train_date_max': str(df.iloc[train_idx][date_col].max().date()),
    'validation_date_min': str(df.iloc[val_idx][date_col].min().date()),
    'validation_date_max': str(df.iloc[val_idx][date_col].max().date()),
    'test_date_min': str(df.iloc[test_idx][date_col].min().date()),
    'test_date_max': str(df.iloc[test_idx][date_col].max().date()),
    'n_embedding_cols': int(len(embedding_cols)),
    'pca_info': pca_info,
    'feature_group_sizes': {k: len(v) for k, v in feature_groups.items()},
}
with open(OUTPUT_DIR / 'dataset_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print('saved to:', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p.name)

## 9. LaTeX-таблица для раздела 2.8

In [ ]:
latex_cols = ['model', 'feature_group', 'accuracy', 'balanced_accuracy', 'f1', 'roc_auc']
latex_table = metrics[latex_cols].copy()
for c in ['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']:
    latex_table[c] = latex_table[c].round(3)
print(latex_table.to_latex(index=False, escape=False))